# RAG (Retrieval Augmentation Generation)

In [ ]:
# pip install faiss-cpu tiktoken
# pip install -U langchain-community langchain-text-splitters langchain-google-genai faiss-cpu

import os,warnings
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
warnings.filterwarnings("ignore")

env = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\bosch\2026\5_advancedPE\code\config.env"
if load_dotenv(env):
    gemini_key = os.getenv("GEMINI_API_KEY")

In [ ]:
# Step 1: Load the text file
# --------------------------

file = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\bosch\2026\5_advancedPE\dataset\healthyliving.txt"

# Create the loader
loader = TextLoader(file, encoding="utf-8")

# Read the file
# This contains the key "page_content" which has the actual text content of the document
# this will be used to create the embeddings that will be stored in the vector database for RAG
documents = loader.load()

# Actual data from the document
data = documents[0].page_content

In [ ]:
# 2) Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)
print("Chunks created:", len(chunks))

In [ ]:
# 3) Create Gemini embeddings
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", google_api_key=gemini_key)

In [ ]:
# Test the embeddings by creating an embedding for a sample query
# Gemini embeddings have a size of 
emb1 = embeddings.embed_query("computer")
emb2 = embeddings.embed_query("what is the capital of India")

In [ ]:
print("Embeddings created:", f"emb1: {len(emb1)}, emb2: {len(emb2)}")

In [ ]:
# 4) Create the FAISS vector database
vector_store = FAISS.from_documents(documents=chunks,embedding=embeddings)

# Save the Vector store to a local file for later use
# vector_store.save_local("faiss_index")

In [ ]:
# 6) Display retrieved chunks
def PrintResults(results):
    for index, document in enumerate(results, start=1):
        print(f"\nResult {index}")
        print(document.page_content)
        # print("Metadata:", document.metadata)

In [ ]:
# 5) Perform similarity search
query = "What are the benefits of a healthy lifestyle?" 
results = vector_store.similarity_search(query=query,k=5)
PrintResults(results)

In [ ]:
# 5) Perform similarity search
query = "Summarise the benefits of walking" 
results = vector_store.similarity_search(query=query,k=1)
PrintResults(results)

In [ ]:
# Fine tune the generated response from the RAG
# ---------------------------------------------

# 1. Define the LLM
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite",api_key=gemini_key,temperature=0)

# 2. Prompt for refining the initial output
refinement_prompt = PromptTemplate.from_template("""
You are an expert editor.

Improve the following Results.

Requirements:
- Make it clear and professional
- Remove repeated information
- Correct grammatical errors
- Retain all important facts
- Keep the final output within 150 words
- Include a suitable heading

Results:
{results}

Refined output:
""")

# 3. Create output parser
parser = StrOutputParser()


# 4. Create the refinement chain
chain = refinement_prompt | llm | parser

In [ ]:
# 5. Final Output after refinement
final_result = chain.invoke({"results": results})

In [ ]:
# 6. Display both outputs
print("\nREFINED FINAL OUTPUT")
print(final_result)

# RAG with ChromaDB

In [ ]:
import os,warnings
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from nltk import pos_tag, word_tokenize
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
warnings.filterwarnings("ignore")

env = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\bosch\2026\5_advancedPE\code\config.env"
if load_dotenv(env):
    gemini_key = os.getenv("GEMINI_API_KEY")

In [ ]:
file = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\bosch\2026\5_advancedPE\dataset\healthyliving.txt"

# Create the loader
loader = TextLoader(file, encoding="utf-8")
documents = loader.load()
data = documents[0].page_content

# 2) Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)
print("Chunks created:", len(chunks))

# 3) Create Gemini embeddings
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", google_api_key=gemini_key)

In [ ]:
# # Test the embeddings by creating an embedding for a sample query

# # Gemini embeddings have a size of 
# emb1 = embeddings.embed_query("computer")
# emb2 = embeddings.embed_query("what is the capital of India")

In [ ]:
# os.getcwd()

# ChromaDB stores the vector database locally.
persist_directory = "./chroma_db"
print(f"ChromaDB persistence directory: {persist_directory}")

# Create a persistent ChromaDB client.
chroma_client = chromadb.PersistentClient(path=persist_directory)
print(chroma_client)

In [ ]:
# Display existing ChromaDB collections.
print(chroma_client.list_collections())

In [ ]:
# Create a new collection in ChromaDB for storing the embeddings.
collection_name = "healthyliving"

In [ ]:
existing_names = [
    item.name if hasattr(item, "name") else str(item)
    for item in chroma_client.list_collections()
]

print(f"Existing collections: {existing_names}")

In [ ]:
# Delete the existing collection so the notebook starts clean.

if collection_name in existing_names:
    chroma_client.delete_collection(name=collection_name)
    print(f"Deleted existing collection: {collection_name}")
else:
    print(f"No existing collection named '{collection_name}' found. Proceeding to create a new one.")

In [ ]:
# HSNW: Hierarchical Small Navigable World
# specifies how ChromaDB measures the similarity (or distance) between embeddings. It is passed as metadata when creating the collection.

# l2:       Eucledian distance      General numeric vectors, image features
# cosine:   Cosine distance         Text embeddings (OpenAI, Sentence Transformers, BERT)
# ip:       Inner product           Specialized embedding models

In [ ]:
if collection_name in existing_names:
    print(f"Collection {collection_name} already exists. Skipping creation.")
    collection = chroma_client.get_collection(name=collection_name)
else:
    # Create a collection using Euclidean distance.
    collection = chroma_client.create_collection(name=collection_name, metadata={"hnsw:space": "l2"} )
    print(f"Created collection: {collection.name}")

In [ ]:
# print(f"Collection count: {collection.count()}")

if collection.count() <= 0:
    print(f"Collection '{collection_name}' is empty.")
else:
    print(f"Collection '{collection_name}' has {collection.count()} items.")

In [ ]:
# Prepare IDs, embeddings, documents, and metadata for ChromaDB.
if collection.count() <= 0:
    ids = []
    document_embeddings = []
    documents = []
    metadatas = []
    
    for i, chunk in enumerate(chunks):
        desc = " ".join(chunk.page_content.split())

        # Create some metadata for each chunk. Here, we are taking some Noun words only
        wd_tokens = word_tokenize(desc)
        pos_token = pos_tag(wd_tokens)
        kw = [word for word,pos in pos_token if pos in ['NN'] ]

        # Create Embeddings for each Chunk
        doc_emb = embeddings.embed_query(desc)

        ids.append(str(i))
        document_embeddings.append(doc_emb)
        documents.append(desc)
        metadatas.append({"chunk_index": str(i), "source": "healthyliving.txt", "keywords":kw })

In [ ]:
print(f"Prepared {len(ids)} records for insertion into collection '{collection_name}'.")

In [ ]:
# Add (Upsert) the records into ChromaDB.
if collection.count() <= 0:
    collection.add(ids=ids, embeddings=document_embeddings, documents=documents, metadatas=metadatas)
    print(f"Inserted {len(ids)} records into collection '{collection_name}'.")
else:
    print(f"Collection {collection_name} already has data. Skipping data insertion.")

In [ ]:
print(f"Records stored: {collection.count()}")

In [ ]:
# Fetch a record by ID.
collection.get(ids=["1"], include=["documents", "metadatas", "embeddings"] )

In [ ]:
# Query Embeddings

query = "Benefits of yoga on health"
query_vector = embeddings.embed_query(query)

In [ ]:
len(query_vector)

In [ ]:
# Retrieve the top 3 closest records.
top = 1

query_response = collection.query(query_embeddings=[query_vector],n_results=top, include=["documents", "metadatas", "distances"])

In [ ]:
# Fine tune the generated response from the RAG
# ---------------------------------------------

# 1. Define the LLM
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite",api_key=gemini_key,temperature=0)

# 2. Prompt for refining the initial output
refinement_prompt = PromptTemplate.from_template("""
You are an expert editor.

Improve the following Results.

Requirements:
- Make it clear and professional
- Remove repeated information
- Correct grammatical errors
- Retain all important facts
- Keep the final output within 150 words
- Include a suitable heading

Results:
{results}

Refined output:
""")

# 3. Create output parser
parser = StrOutputParser()


# 4. Create the refinement chain
chain = refinement_prompt | llm | parser

In [ ]:
results = query_response['documents'][0][0]

final_result = chain.invoke({"results":results})

In [ ]:
final_result

# Read an existing ChromaDB collection
# With L2 distance, smaller values indicate greater similarity.

In [ ]:
# Path containing the existing Chroma database
client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_collection(name="healthyliving")

print("Collection name:", collection.name)
print("Number of records:", collection.count())

# Create Gemini embeddings
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", google_api_key=gemini_key)

In [ ]:
# Fetch a record by ID.
# collection.get(ids=["1"], include=["documents", "metadatas", "embeddings"] )


# Search using metadata
result = collection.get(
    where={"keywords": {"$contains":"health"}},
    limit=2,
    include=["documents", "metadatas"]
)

In [ ]:
result

In [ ]:
sample = collection.peek(limit=2)

for record_id, metadata in zip(
    sample["ids"],
    sample["metadatas"]
):
    print("ID:", record_id)
    print("Keywords:", metadata.get("keywords"))
    print("Full metadata:", metadata)
    print()

In [ ]:
result

In [ ]:
# Query Embeddings
query = "Benefits of good mental health"
query_vector = embeddings.embed_query(query)

In [ ]:
# Retrieve the top 3 closest records.
top = 3
query_response = collection.query(query_embeddings=[query_vector],n_results=top, include=["documents", "metadatas", "distances"])

In [ ]:
query_response

In [ ]:
# query_response['ids']
query_response['distances']
# query_response['documents']